# Tutorial 1: MetabolicModel and data loading

This notebook introduces the three objects that usually come first in a Metabis workflow:

- `MetabolicModel`: the packaged reference model and its identifiers
- `ReactionData`: the reaction activity matrix aligned to the model
- `ExpressionData`: the gene expression matrix translated to the model gene identifiers

The goal is to load each object, inspect the validation output, and see the minimum `DataManager` state before grouping or statistical analysis.

## Before you run it

Run this notebook from an editable Metabis install. The bundled Th17 Compass example files are loaded from `metabis/data`.

In [1]:
import sys
from pathlib import Path



def find_metabis_repo_root_for_import(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (candidate / "src" / "metabis").exists():
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


repo_root_for_import = find_metabis_repo_root_for_import(Path.cwd().resolve())
src_path = repo_root_for_import / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from metabis import DataManager, ExpressionData, MetabolicModel, ReactionData
from metabis.version import __version__ as metabis_version

/inco/home01/g/gdufort/.conda/envs/metabis_test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
mkdir -p failed for path /home/gdufort/.cache/matplotlib: [Errno 13] Permission denied: '/home/gdufort'
Matplotlib created a temporary cache directory at /tmp/matplotlib-yhttilqi because there was an issue with the default path (/home/gdufort/.cache/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
def find_metabis_repo_root(start: Path) -> Path:
    direct_candidates = [start, *start.parents]
    nested_candidates = [candidate / "metabis" for candidate in direct_candidates]

    for candidate in [*direct_candidates, *nested_candidates]:
        if (
            (candidate / "pyproject.toml").exists()
            and (candidate / "demo").exists()
            and (candidate / "src" / "metabis").exists()
        ):
            return candidate

    raise FileNotFoundError("Could not locate the packaged Metabis repository root.")


def resolve_example_data_dir(repo_root: Path) -> Path:
    data_dir = repo_root / "data"

    if (
        (data_dir / "Th17_Compass_reactions.csv").exists()
        and (data_dir / "Th17_Compass_expression.csv").exists()
    ):
        return data_dir

    raise FileNotFoundError(
        "Could not find the bundled Th17 Compass example files in metabis/data."
    )


repo_root = find_metabis_repo_root(Path.cwd().resolve())
data_dir = resolve_example_data_dir(repo_root)
reaction_data_path = data_dir / "Th17_Compass_reactions.csv"
expression_data_path = data_dir / "Th17_Compass_expression.csv"

print(f"Metabis version: {metabis_version}")
print(f"Repository root: {repo_root}")
print(f"Data directory: {data_dir}")

Metabis version: 0.0.1
Repository root: /data/gdufort/metabis/metabis
Data directory: /data/gdufort/metabis/metabis/data


## 1. Initialize the metabolic model

`MetabolicModel` exposes the packaged Human1-based reference used throughout the library. It is the object that knows about reactions, genes, subsystems, compartments, and identifier translation.

For most notebook workflows, the main things you do first are:

- build the model with caching enabled
- apply any identifier alignment corrections needed for your input data
- inspect the reaction, gene, and subsystem spaces that later objects will use

In [3]:
model = MetabolicModel(use_cache=True)

alignment_file_path = model.get_alignment_file_path("RECON2")
alignment_results = model.apply_alignment_corrections(alignment_file_path)

print(
    f"Applied {alignment_results['corrections_applied']} reaction ID corrections from {Path(alignment_file_path).name}."
)
print(f"Model reactions: {len(model.get_reactions())}")
print(f"Model genes: {len(model.get_genes())}")
print(f"Model subsystems: {len(model.get_subsystems())}")

Applied 233 reaction ID corrections from alignment_RECON2.txt.
Model reactions: 12971
Model genes: 2887
Model subsystems: 146


In [4]:
example_subsystem = model.get_subsystems()[0]

print("Example reactions:", model.get_reactions()[:5])
print("Example genes:", model.get_genes()[:5])
print("Example subsystem:", example_subsystem)
print(
    "Example subsystem size:",
    len(model.get_reactions_for_subsystem(example_subsystem)),
)
print(
    "SAT1 translated to model gene ID:",
    model.translate_gene(
        "SAT1",
        input_format="geneSymbols",
        output_format="ensmbl_id",
    ),
)

Example reactions: ['MAR03905', 'MAR03907', 'MAR04097', 'MAR04099', 'MAR04108']
Example genes: ['ENSG00000000419', 'ENSG00000001036', 'ENSG00000001084', 'ENSG00000001630', 'ENSG00000002549']
Example subsystem: glycolysis_gluconeogenesis
Example subsystem size: 44
SAT1 translated to model gene ID: ENSG00000130066


## 2. Load the reaction matrix

`ReactionData` is responsible for loading a reaction activity table and translating its identifiers into the reaction space used by the model.

For the Compass example, three details matter:

- the file is tab-delimited even though it uses a `.csv` suffix
- samples are stored in columns
- reversible reactions are split into `_pos` and `_neg`, so the data is loaded in `dual` mode

In [5]:
reaction_loading_options = {
    "data_mode": "dual",
    "reaction_name_format": "rxnRecon3DID",
    "csv_separator": "\t",
    "samples_in": "columns",
    "check_old_identifiers": True,
}

expression_loading_options = {
    "gene_format": "geneSymbols",
    "normalization_options": ["CPM"],
    "data_format": "csv",
    "csv_separator": "\t",
    "samples_in": "columns",
}

reaction_loading_options, expression_loading_options

({'data_mode': 'dual',
  'reaction_name_format': 'rxnRecon3DID',
  'csv_separator': '\t',
  'samples_in': 'columns',
  'check_old_identifiers': True},
 {'gene_format': 'geneSymbols',
  'normalization_options': ['CPM'],
  'data_format': 'csv',
  'csv_separator': '\t',
  'samples_in': 'columns'})

In [6]:
reaction_data, reaction_validation = ReactionData.create_with_validation(
    str(reaction_data_path),
    metabolic_model=model,
    **reaction_loading_options,
)

reaction_df = reaction_data.get_reaction_df()
reaction_alignment = reaction_validation.get("alignment_report", {})

print(
    f"Reaction matrix: {reaction_df.shape[0]} samples x {reaction_df.shape[1]} reaction features"
)
print(f"Mapped reactions: {reaction_alignment.get('mapped_count', 0)}")
print(f"Unmapped reactions: {reaction_alignment.get('unmapped_count', 0)}")
print(f"Model coverage: {reaction_alignment.get('model_coverage_percent', 0):.2f}%")
reaction_validation

Reaction matrix: 290 samples x 8462 reaction features
Mapped reactions: 5931
Unmapped reactions: 1010
Model coverage: 45.31%


{'status': 'success',
 'messages': ['Reaction data loaded successfully'],
 'alignment_report': {'mapped_count': 5931,
  'unmapped_count': 1010,
  'base_reactions_mapped': 5877,
  'total_model_reactions': 12971,
  'model_coverage_percent': 45.30876570811811,
  'subsystems_with_reactions': 121,
  'total_subsystems': 146}}

In [7]:
print("First sample IDs:", reaction_data.get_sample_ids()[:5])
reaction_df.iloc[:3, :5]

First sample IDs: ['Th17p_SRR2921282', 'Th17p_SRR2921283', 'Th17p_SRR2921284', 'Th17p_SRR2921285', 'Th17p_SRR2921286']


,MAR07915,MAR07914,MAR07918,MAR07917,MAR07924
Th17p_SRR2921282,3.558842,3.234259,3.408527,3.208960,3.319563
Th17p_SRR2921283,3.575600,3.243618,3.431394,3.226069,3.345651
Th17p_SRR2921284,3.488729,3.277223,3.312902,3.221700,3.238742


## 3. Load the expression matrix

`ExpressionData` converts an expression table into the model gene space. In this example, the input file uses gene symbols, while the model stores Ensembl-like identifiers internally.

This class also handles normalization options, so it is the right place to keep basic preprocessing tied to the expression matrix itself.

In [8]:
expression_data, expression_validation = ExpressionData.create_with_validation(
    str(expression_data_path),
    gene_list=model.get_genes(),
    metabolic_model=model,
    **expression_loading_options,
)

expression_df = expression_data.get_expression_df()
expression_alignment = expression_validation.get("alignment_report", {})

print(
    f"Expression matrix: {expression_df.shape[0]} samples x {expression_df.shape[1]} genes"
)
print(f"Mapped genes: {expression_alignment.get('mapped_count', 0)}")
print(f"Unmapped genes: {expression_alignment.get('unmapped_count', 0)}")
print(f"Model coverage: {expression_alignment.get('model_coverage_percent', 0):.2f}%")
expression_validation

Expression matrix: 290 samples x 1532 genes
Mapped genes: 1532
Unmapped genes: 7281
Model coverage: 53.07%


{'status': 'success',
 'messages': ['Expression data loaded successfully'],
 'alignment_report': {'total_input_genes': 8813,
  'mapped_count': 1532,
  'unmapped_count': 7281,
  'total_model_genes': 2887,
  'model_coverage_percent': 53.06546588153793}}

In [9]:
print("First sample IDs in expression data:", expression_df.index[:5].tolist())
expression_df.iloc[:3, :5]

First sample IDs in expression data: ['Th17p_SRR2921282', 'Th17p_SRR2921283', 'Th17p_SRR2921284', 'Th17p_SRR2921285', 'Th17p_SRR2921286']


,ENSG00000244038,ENSG00000090432,ENSG00000127481,ENSG00000159423,ENSG00000117115
Th17p_SRR2921282,154.211895,33.576679,40.579640,0.0,0.000000
Th17p_SRR2921283,129.692931,13.641277,38.585884,0.0,10.171181
Th17p_SRR2921284,188.599246,67.380472,27.605459,0.0,11.892628


## 4. Create the minimal DataManager

`DataManager` is the orchestration object used by the rest of Metabis. At this stage it simply holds the model and the loaded matrices. Later notebooks will add groups, experiment analysis, and group comparison objects to it.

In [10]:
dm = DataManager(model, reaction_data, expression_data, None)

shared_samples = sorted(set(reaction_data.get_sample_ids()) & set(expression_df.index))

print(f"Shared sample IDs across matrices: {len(shared_samples)}")
print(f"Current mode: {dm.get_mode()}")
print(f"Available modes: {dm.get_available_modes()}")
print(dm.get_mode_description())

Shared sample IDs across matrices: 290
Current mode: DataManagerMode.NO_EXPERIMENT
Available modes: [<DataManagerMode.NO_EXPERIMENT: 'no_experiment'>]
NO_EXPERIMENT mode: Basic metadata only, no statistical analysis or experimental data coloring


## Takeaways

At this point you have the minimum state required for the rest of the library:

- a model with identifier translation and subsystem structure
- a validated reaction matrix
- a validated expression matrix
- a `DataManager` ready for grouping

The next notebook builds on the same setup and introduces groups plus dimensionality-reduction plots.